# 🏢 Anticipez les besoins en consommations de bâtiments

## 📋 Présentation du Projet
Dans le cadre de son engagement pour la **neutralité carbone d'ici 2050**, la ville de Seattle analyse les données de consommation énergétique de ses bâtiments non résidentiels.

* **Objectif :** Prédire les émissions de CO2 et la consommation totale d'énergie.
* **Cible :** Bâtiments non destinés à l'habitation.

---

##  1. Imports des modules et du dataset


### Stack Logicielle
| Modules | Version | Utilité |
| :--- | :---: | :--- |
| `Python` | `3.14` | Langage principal  |
| `Matplotlib` | `3.10.8` | Création et visualisations de graphiques  |
| `Pandas` | `3.0.1` | Manipulation des données |
| `Seaborn` | `0.13.2` | Visualisation statistiques |
| `Sickit-learn` | `Texte` | Modelisation |   **A completer**

### Source des données
* **Fichier** : `../data/raw/2016_Building_Energy_Benchmarking.csv`
*  **Format** : CSV

---

##  2. Analyse de la structure des données

###  Dimensions (Shape)
L'appel à `shape` indique que le dataset contient :
* **3 376** lignes
* **46** colonnes

**Question** : si j'ai mis un head() dans l'analyse, faut-il forcément le reporter ici ? seulement si quelque chose de notable à relever ?

###  Qualité des données (Info)
L'analyse via `info()` montre que plusieurs colonnes comptent un nombre critique de valeurs nulles :
* `Comments`
* `Outlier`
* `YearsENERGYSTARCertified`
* `ThirdLargestPropertyUseType`

La colonne `ListOfAllPropertyUseTypes` sera difficilement exploitable par le modèle (liste sous forme de string).


###  Statistiques descriptives (Describe)

| Variables | Observations | Déductions |
| :--- | :--- | :--- |
| `NumberofBuildings` | 75% à 1 et Max à 111 | Présence d'**outliers** marqués. |
| `NumberofFloors` | Max à 99, alors que 75% sont à 5 | Présence potentielle d'**outliers**. |
| `Electricity(kBtu)` | Valeur min négative (`-1.154e+05`) | **Valeurs aberrantes** à corriger. |

> **Note sur le nettoyage :** Les variables `DataYear`, `Comments`, `City` et `States` disposent d'une valeur unique ou sont vides. Elles ne présentent aucune utilité statistique et seront supprimées lors de l'étape de nettoyage.

**Questions**

Utile de reporter des observations plus annexes ?

- SteamUse : + 75% des batiments n'utilisent pas de gaz
- PropertyGFAParking : + 75% des batiments n'ont pas de parking

J'imagine en fonction de si l'info sert plus tard dans le notebook

---

##  3. Analyse des données (fond) :
| Variables | Pourcentage de valeurs vides | Déductions |
| :--- | :---: | :--- |
| `Outliers` | 99.052133% | Outliers connus du dataset -> suppression des lignes avec une valeurs puis suppression de la colonne |
| `Comments` | 100% | Colonne vide -> à supprimer |
| `YearsENERGYSTARCertified` | 96.475118% | Trop peu de valeurs -> à supprimer |
| ``ThirdLargestPropertyUseType`` | 82.345972% | Trop peu de valeurs mais utile via feature engineering |
| ``ThirdLargestPropertyUseTypeGFA`` | 82.345972% | Trop peu de valeurs mais utile via feature engineering |
| `DefaultData` | 96.652844% | Faible pourcentage de valeurs entrée par défaut donc on peut supprimer |

**Questions**
Quel schéma peut être pertinent ici ? Redondance en plus des pourcentage ?
Reporter le schéma ici ?

---

## 4. Suppression des lignes et des colonnes inutiles

### Cible
Notre variable cible sera `SiteEnergyUseWN(kBtu)` :
- L'objectif de l'analyse est d'anticiper la consommation energetique ce qui correspond à cette variable
- Mesure en kBtu et WN qui sont des indicateurs universelles
- La colonne est 



### Filtrage des lignes
**Filtrage par ligne**
- L'étude porte sur les batiments non résidentiel. On ne conservera donc que les lignes de _BuildingType_ n'ayant pas la mention "Multifamily" dans leur libellé.
- On supprime les lignes ayant un outlier connu du dataset en se basant sur les valeurs non vides de _Outlier_.
- Suppression des lignes avec des valeurs par défaut ajoutée DefaultData

**A faire** : filtrer les lignes vides de **SiteEnergyUseWN(kBtu)**


**Question**


### Filtrage des colonnes
**Post analyse exploratoire**

Colonnes qui se révèlent inutile suite à l'analyse exploratoire
| Variables | Justification |
| :--- | :--- |
| `Outliers` | Post-filtre, ne dispose plus que de valeurs vides |
| `Comments` | Colonne vide |
| `DataYear` | Valeur unique |
| `City` | Valeur unique |
| `States` | Valeur unique |
|`YearsENERGYSTARCertified`|Trop peu de valeurs|
|`ListOfAllPropertyUseTypes`|Difficilement exploitable pour le modèle|

**Carateristiques du batiment**
Plusieurs variables de localisation et d'identification de batiment redondantes.

Choix de garder latitude et longitude :
- Plus simple à traiter que des strings pour les adresses
- Meilleur précision geographique
- L'indice du dataframe nous permettra d'identifier chaque batiment
- `ComplianceStatus` n'a aucune valeur pour la prediction

_Suppression des variables de localisation_
| Variables | Justification |
| :--- | :--- |
| `Address` | Redondance de localisation |
| `ZipCode` | Redondance de localisation |
| `TaxParcelIdentificationNumber` | Redondance de localisation |
| `OSEBuildingID` | Redondance d'identification' |
| `ComplianceStatus` | Aucune utilité pour le modèle|


**Unite de mesure**

Beaucoup d'unité de mesure sont des composantes de notre variable cible


Plusieurs mesures ont leurs equivalent en différentes unités
Plusieurs mesures ont leurs équivalent brut et WN (Weather Normalize).

Choix de ne garder que les données
Le kBtu étant une mesure universelle, on fait le choix de conserver les valeurs energetique adapter à cette unité de mesure

Choix de ne garder que les données de mesure WN :
- Plus exploitable de comparer les mesures selon la même unité
- Plus fiable pour l'évaluation strict de la consommation du batiment (ajusté à la météo)

| Variables | Justification |
| :--- | :--- |
| `Electricity(kWh)` | Equivalent existant WN |
| `NaturalGas(therms)` | Equivalent existant WN |
| `SiteEnergyUse(kBtu)` | Equivalent existant WN |
| `SiteEUI(kBtu/sf) ` | Equivalent existant WN |
| `SourceEUI(kBtu/sf)` | Equivalent existant WN |
| `SiteEnergyUse(kBtu)` | Equivalent existant WN |
| `NaturalGas(kBtu)` | Composantes de la variable cible |
| `SteamUse(kBtu)` | Composantes de la variable cible |
| `Electricity(kBtu)` | Composantes de la variable cible |
| `TotalGHGEmissions` | Calculé à partir de la variable cible|
| `GHGEmissionsIntensity` | Calculé à partir de la variable cible |




Shape post traitement

**Questions**
- A quel point faut-il argumenter ici dans le choix des colonnes à conserver ?
- Garder OSEBuildingID pour identifier le batiment ? ou l'indice du dataframe est suffisant ?


## 5. Traitement des valeurs aberrantes et impossibles
Valeurs impossibles,  (ex: yearbuilt = 2040)
Valeurs aberrantes, à traiter/visualiser avec Boxplot
Arbitrage de ce qu'on fait pour ces valeurs là (remplacement, suppression)

A l'issue, plus de valeurs inutiles et vides

## 6. Analyse univarié
Analyse univarié, sur une variable (voir lien compte rendu)

## 7. Analyse bivarié
Analyse bivarié, sur plusieurs features (trouver les corrélations)

Dégager des hypothèses à partir des différentes représentations et graphiques

Encodage des données pour la modélisation : On ne peut pas faire d'algorithme de classification sur des string, il faut donc les convertir en donnés utilisable

Feature engineering, créer une nouvelle variable à partir d'autres variables existantes

DataSet prêt à l'emploi et à sauvegarder


##  1. Imports des modules et du dataset


In [790]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import dis

In [791]:
building_consumption = pd.read_csv("../data/raw/2016_Building_Energy_Benchmarking.csv")

## 2. Analyse de la structure des données

In [792]:
# Dimension du dataset
building_consumption.shape

(3376, 46)

In [793]:
# Première aperçu du dataset
building_consumption.head()

,OSEBuildingID,DataYear,BuildingType,PrimaryPropertyType,PropertyName,Address,City,State,ZipCode,TaxParcelIdentificationNumber,CouncilDistrictCode,Neighborhood,Latitude,Longitude,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFATotal,PropertyGFAParking,PropertyGFABuilding(s),ListOfAllPropertyUseTypes,LargestPropertyUseType,LargestPropertyUseTypeGFA,SecondLargestPropertyUseType,SecondLargestPropertyUseTypeGFA,ThirdLargestPropertyUseType,ThirdLargestPropertyUseTypeGFA,YearsENERGYSTARCertified,ENERGYSTARScore,SiteEUI(kBtu/sf),SiteEUIWN(kBtu/sf),SourceEUI(kBtu/sf),SourceEUIWN(kBtu/sf),SiteEnergyUse(kBtu),SiteEnergyUseWN(kBtu),SteamUse(kBtu),Electricity(kWh),Electricity(kBtu),NaturalGas(therms),NaturalGas(kBtu),DefaultData,Comments,ComplianceStatus,Outlier,TotalGHGEmissions,GHGEmissionsIntensity
0,1,2016,NonResidential,Hotel,Mayflower park hotel,405 Olive way,Seattle,WA,98101.0,0659000030,7,DOWNTOWN,47.61220,-122.33799,1927,1.0,12,88434,0,88434,Hotel,Hotel,88434.0,NaN,NaN,NaN,NaN,NaN,60.0,81.699997,84.300003,182.500000,189.000000,7226362.5,7456910.0,2003882.00,1.156514e+06,3946027.0,12764.52930,1276453.0,False,NaN,Compliant,NaN,249.98,2.83
1,2,2016,NonResidential,Hotel,Paramount Hotel,724 Pine street,Seattle,WA,98101.0,0659000220,7,DOWNTOWN,47.61317,-122.33393,1996,1.0,11,103566,15064,88502,"Hotel, Parking, Restaurant",Hotel,83880.0,Parking,15064.0,Restaurant,4622.0,NaN,61.0,94.800003,97.900002,176.100006,179.399994,8387933.0,8664479.0,0.00,9.504252e+05,3242851.0,51450.81641,5145082.0,False,NaN,Compliant,NaN,295.86,2.86
2,3,2016,NonResidential,Hotel,5673-The Westin Seattle,1900 5th Avenue,Seattle,WA,98101.0,0659000475,7,DOWNTOWN,47.61393,-122.33810,1969,1.0,41,956110,196718,759392,Hotel,Hotel,756493.0,NaN,NaN,NaN,NaN,NaN,43.0,96.000000,97.699997,241.899994,244.100006,72587024.0,73937112.0,21566554.00,1.451544e+07,49526664.0,14938.00000,1493800.0,False,NaN,Compliant,NaN,2089.28,2.19
3,5,2016,NonResidential,Hotel,HOTEL MAX,620 STEWART ST,Seattle,WA,98101.0,0659000640,7,DOWNTOWN,47.61412,-122.33664,1926,1.0,10,61320,0,61320,Hotel,Hotel,61320.0,NaN,NaN,NaN,NaN,NaN,56.0,110.800003,113.300003,216.199997,224.000000,6794584.0,6946800.5,2214446.25,8.115253e+05,2768924.0,18112.13086,1811213.0,False,NaN,Compliant,NaN,286.43,4.67
4,8,2016,NonResidential,Hotel,WARWICK SEATTLE HOTEL (ID8),401 LENORA ST,Seattle,WA,98121.0,0659000970,7,DOWNTOWN,47.61375,-122.34047,1980,1.0,18,175580,62000,113580,"Hotel, Parking, Swimming Pool",Hotel,123445.0,Parking,68009.0,Swimming Pool,0.0,NaN,75.0,114.800003,118.699997,211.399994,215.600006,14172606.0,14656503.0,0.00,1.573449e+06,5368607.0,88039.98438,8803998.0,False,NaN,Compliant,NaN,505.01,2.88


In [794]:
# Première aperçu de la qualité des données
building_consumption.info()

<class 'pandas.DataFrame'>
RangeIndex: 3376 entries, 0 to 3375
Data columns (total 46 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   OSEBuildingID                    3376 non-null   int64  
 1   DataYear                         3376 non-null   int64  
 2   BuildingType                     3376 non-null   str    
 3   PrimaryPropertyType              3376 non-null   str    
 4   PropertyName                     3376 non-null   str    
 5   Address                          3376 non-null   str    
 6   City                             3376 non-null   str    
 7   State                            3376 non-null   str    
 8   ZipCode                          3360 non-null   float64
 9   TaxParcelIdentificationNumber    3376 non-null   str    
 10  CouncilDistrictCode              3376 non-null   int64  
 11  Neighborhood                     3376 non-null   str    
 12  Latitude                       

In [795]:
building_consumption.describe(include='all')

,OSEBuildingID,DataYear,BuildingType,PrimaryPropertyType,PropertyName,Address,City,State,ZipCode,TaxParcelIdentificationNumber,CouncilDistrictCode,Neighborhood,Latitude,Longitude,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFATotal,PropertyGFAParking,PropertyGFABuilding(s),ListOfAllPropertyUseTypes,LargestPropertyUseType,LargestPropertyUseTypeGFA,SecondLargestPropertyUseType,SecondLargestPropertyUseTypeGFA,ThirdLargestPropertyUseType,ThirdLargestPropertyUseTypeGFA,YearsENERGYSTARCertified,ENERGYSTARScore,SiteEUI(kBtu/sf),SiteEUIWN(kBtu/sf),SourceEUI(kBtu/sf),SourceEUIWN(kBtu/sf),SiteEnergyUse(kBtu),SiteEnergyUseWN(kBtu),SteamUse(kBtu),Electricity(kWh),Electricity(kBtu),NaturalGas(therms),NaturalGas(kBtu),DefaultData,Comments,ComplianceStatus,Outlier,TotalGHGEmissions,GHGEmissionsIntensity
count,3376.000000,3376.0,3376,3376,3376,3376,3376,3376,3360.000000,3376,3376.000000,3376,3376.000000,3376.000000,3376.000000,3368.000000,3376.000000,3.376000e+03,3376.000000,3.376000e+03,3367,3356,3.356000e+03,1679,1679.000000,596,596.000000,119.0,2533.000000,3369.000000,3370.000000,3367.000000,3367.000000,3.371000e+03,3.370000e+03,3.367000e+03,3.367000e+03,3.367000e+03,3.367000e+03,3.367000e+03,3376,0.0,3376,32,3367.000000,3367.000000
unique,NaN,NaN,8,24,3362,3354,1,1,NaN,3268,NaN,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,466,56,NaN,50,NaN,44,NaN,65.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,4,2,NaN,NaN
top,NaN,NaN,NonResidential,Low-Rise Multifamily,Northgate Plaza,2600 SW Barton St,Seattle,WA,NaN,1625049001,NaN,DOWNTOWN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Multifamily Housing,Multifamily Housing,NaN,Parking,NaN,Retail Store,NaN,2016.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,Compliant,Low outlier,NaN,NaN
freq,NaN,NaN,1460,987,3,4,3376,3376,NaN,8,NaN,573,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,866,1667,NaN,976,NaN,110,NaN,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3263,NaN,3211,23,NaN,NaN
mean,21208.991114,2016.0,NaN,NaN,NaN,NaN,NaN,NaN,98116.949107,NaN,4.439277,NaN,47.624033,-122.334795,1968.573164,1.106888,4.709123,9.483354e+04,8001.526066,8.683201e+04,NaN,NaN,7.917764e+04,NaN,28444.075817,NaN,11738.675166,NaN,67.918674,54.732116,57.033798,134.232848,137.783932,5.403667e+06,5.276726e+06,2.745959e+05,1.086639e+06,3.707612e+06,1.368505e+04,1.368505e+06,NaN,NaN,NaN,NaN,119.723971,1.175916
std,12223.757015,0.0,NaN,NaN,NaN,NaN,NaN,NaN,18.615205,NaN,2.120625,NaN,0.047758,0.027203,33.088156,2.108402,5.494465,2.188376e+05,32326.723928,2.079398e+05,NaN,NaN,2.017034e+05,NaN,54392.917928,NaN,29331.199286,NaN,26.873271,56.273124,57.163330,139.287554,139.109807,2.161063e+07,1.593879e+07,3.912173e+06,4.352478e+06,1.485066e+07,6.709781e+04,6.709781e+06,NaN,NaN,NaN,NaN,538.832227,1.821452
min,1.000000,2016.0,NaN,NaN,NaN,NaN,NaN,NaN,98006.000000,NaN,1.000000,NaN,47.499170,-122.414250,1900.000000,0.000000,0.000000,1.128500e+04,0.000000,3.636000e+03,NaN,NaN,5.656000e+03,NaN,0.000000,NaN,0.000000,NaN,1.000000,0.000000,0.000000,0.000000,-2.100000,0.000000e+00,0.000000e+00,0.000000e+00,-3.382680e+04,-1.154170e+05,0.000000e+00,0.000000e+00,NaN,NaN,NaN,NaN,-0.800000,-0.020000
25%,19990.750000,2016.0,NaN,NaN,NaN,NaN,NaN,NaN,98105.000000,NaN,3.000000,NaN,47.599860,-122.350662,1948.000000,1.000000,2.000000,2.848700e+04,0.000000,2.775600e+04,NaN,NaN,2.509475e+04,NaN,5000.000000,NaN,2239.000000,NaN,53.000000,27.900000,29.400000,74.699997,78.400002,9.251286e+05,9.701822e+05,0.000000e+00,1.874229e+05,6.394870e+05,0.000000e+00,0.000000e+00,NaN,NaN,NaN,NaN,9.495000,0.210000
50%,23112.000000,2016.0,NaN,NaN,NaN,NaN,NaN,NaN,98115.000000,NaN,4.000000,NaN,47.618675,-122.332495,1975.000000,1.000000,4.000000,4.417500e+04,0.000000,4.321600e+04,NaN,NaN,3.989400e+04,NaN,10664.000000,NaN,5043.000000,NaN,75.000000,38.599998,40.900002,96.199997,101.099998,1.803753e+06,1.904452e+06,0.000000e+00,3.451299e+05,1.177583e+06,3.237538e+03,3.237540e+05,NaN,NaN,NaN,NaN,33.920000,0.610000
75%,25994.250000,2016.0,NaN,NaN,NaN,NaN,NaN,NaN,98122.000000,NaN,7.000000,NaN,4

In [796]:
building_consumption["BuildingType"].value_counts()

BuildingType
NonResidential          1460
Multifamily LR (1-4)    1018
Multifamily MR (5-9)     580
Multifamily HR (10+)     110
SPS-District K-12         98
Nonresidential COS        85
Campus                    24
Nonresidential WA          1
Name: count, dtype: int64

## 3. Analyse des données (fond) 

In [797]:
repartition_outlier = building_consumption["Outlier"].value_counts(normalize=True, dropna=False) * 100
display(repartition_outlier)


repartition_comments = building_consumption["Comments"].value_counts(normalize=True, dropna=False) * 100
display(repartition_comments)

repartition_YearsENERGYSTARCertified = building_consumption["YearsENERGYSTARCertified"].value_counts(normalize=True, dropna=False) * 100
display(repartition_YearsENERGYSTARCertified)

repartition_ThirdLargestPropertyUseType = building_consumption["ThirdLargestPropertyUseType"].value_counts(normalize=True, dropna=False) * 100
display(repartition_ThirdLargestPropertyUseType)

repartition_defaultData = building_consumption["DefaultData"].value_counts(normalize=True, dropna=False) * 100
display(repartition_defaultData)


Outlier
NaN             99.052133
Low outlier      0.681280
High outlier     0.266588
Name: proportion, dtype: float64

Comments
NaN    100.0
Name: proportion, dtype: float64

YearsENERGYSTARCertified
NaN                                 96.475118
2016                                 0.414692
20172016                             0.236967
2017                                 0.207346
2014                                 0.177725
                                      ...    
2011                                 0.029621
20172015201420132011                 0.029621
20152012                             0.029621
20162015201420132012201020092008     0.029621
20162015201020092008                 0.029621
Name: proportion, Length: 66, dtype: float64

ThirdLargestPropertyUseType
NaN                                                     82.345972
Retail Store                                             3.258294
Office                                                   3.110190
Parking                                                  2.103081
Restaurant                                               1.658768
Other                                                    1.451422
Swimming Pool                                            0.859005
Non-Refrigerated Warehouse                               0.533175
Medical Office                                           0.503555
Data Center                                              0.414692
Multifamily Housing                                      0.355450
Food Service                                             0.325829
Social/Meeting Hall                                      0.325829
Other - Restaurant/Bar                                   0.266588
Pre-school/Daycare                              

DefaultData
False    96.652844
True      3.347156
Name: proportion, dtype: float64

## 4. Suppression des colonnes inutiles

### Cible

In [810]:
building_consumption["SiteEnergyUseWN(kBtu)"].isna().sum()

np.int64(3)

### Filtrage par ligne

In [ ]:
# Filtre des batiments non destinés à l'habitation
building_consumption = building_consumption[building_consumption["BuildingType"].isin(["NonResidential", "SPS-District K-12", "Nonresidential COS", "Campus", "Nonresidential WA"])].copy()

# Filtre des lignes ayant un outlier identifié
building_consumption = building_consumption[building_consumption["Outlier"].isna()].copy()

#Filtre des lignes ayant des données entrées par défaut
building_consumption = building_consumption[building_consumption["DefaultData"]!= True].copy()


building_consumption = building_consumption[~building_consumption["SiteEnergyUseWN(kBtu)"].isna()].copy()

SiteEnergyUseWN(kBtu)
0.000000e+00    24
7.456910e+06     1
8.664479e+06     1
7.393711e+07     1
6.946800e+06     1
                ..
1.025432e+06     1
1.053706e+06     1
6.053764e+06     1
7.828413e+05     1
1.293722e+06     1
Name: count, Length: 1539, dtype: int64

In [800]:
# Nettoyage des colonnes inutiles
building_consumption = building_consumption[[ 
    #'OSEBuildingID','DataYear', 
       'BuildingType', 'PrimaryPropertyType',
       #'PropertyName', 'Address', 'City', 'State', 'ZipCode', 'TaxParcelIdentificationNumber', 'CouncilDistrictCode', 
       'Neighborhood',
       'Latitude', 'Longitude', 
       'YearBuilt', 'NumberofBuildings',
       'NumberofFloors', 'PropertyGFATotal', 'PropertyGFAParking',
       'PropertyGFABuilding(s)', 'ListOfAllPropertyUseTypes',
       'LargestPropertyUseType', 'LargestPropertyUseTypeGFA',
       'SecondLargestPropertyUseType', 'SecondLargestPropertyUseTypeGFA',
       'ThirdLargestPropertyUseType', 'ThirdLargestPropertyUseTypeGFA',
       #'YearsENERGYSTARCertified', 
       'ENERGYSTARScore', 
       #'SiteEUI(kBtu/sf)',
       #'SiteEUIWN(kBtu/sf)', 
       #'SourceEUI(kBtu/sf)', 
       #'SourceEUIWN(kBtu/sf)',
       #'SiteEnergyUse(kBtu)', 
       'SiteEnergyUseWN(kBtu)', 
       #'SteamUse(kBtu)',
       #'Electricity(kWh)', 
       #'Electricity(kBtu)', 
       #'NaturalGas(therms)',
       #'NaturalGas(kBtu)', 
       # 'DefaultData', 'Comments', 'ComplianceStatus', 'Outlier', 'TotalGHGEmissions', 'GHGEmissionsIntensity'

]].copy()
display(building_consumption)

,BuildingType,PrimaryPropertyType,Neighborhood,Latitude,Longitude,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFATotal,PropertyGFAParking,PropertyGFABuilding(s),ListOfAllPropertyUseTypes,LargestPropertyUseType,LargestPropertyUseTypeGFA,SecondLargestPropertyUseType,SecondLargestPropertyUseTypeGFA,ThirdLargestPropertyUseType,ThirdLargestPropertyUseTypeGFA,ENERGYSTARScore,SiteEnergyUseWN(kBtu)
0,NonResidential,Hotel,DOWNTOWN,47.61220,-122.33799,1927,1.0,12,88434,0,88434,Hotel,Hotel,88434.0,NaN,NaN,NaN,NaN,60.0,7.456910e+06
1,NonResidential,Hotel,DOWNTOWN,47.61317,-122.33393,1996,1.0,11,103566,15064,88502,"Hotel, Parking, Restaurant",Hotel,83880.0,Parking,15064.0,Restaurant,4622.0,61.0,8.664479e+06
2,NonResidential,Hotel,DOWNTOWN,47.61393,-122.33810,1969,1.0,41,956110,196718,759392,Hotel,Hotel,756493.0,NaN,NaN,NaN,NaN,43.0,7.393711e+07
3,NonResidential,Hotel,DOWNTOWN,47.61412,-122.33664,1926,1.0,10,61320,0,61320,Hotel,Hotel,61320.0,NaN,NaN,NaN,NaN,56.0,6.946800e+06
4,NonResidential,Hotel,DOWNTOWN,47.61375,-122.34047,1980,1.0,18,175580,62000,113580,"Hotel, Parking, Swimming Pool",Hotel,123445.0,Parking,68009.0,Swimming Pool,0.0,75.0,1.465650e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3370,Nonresidential COS,Other,DELRIDGE NEIGHBORHOODS,47.54067,-122.37441,1982,1.0,1,18261,0,18261,Other - Recreation,Other - Recreation,18261.0,NaN,NaN,NaN,NaN,NaN,1.025432e+06
3372,Nonresidential COS,Other,DOWNTOWN,47.59625,-122.32283,2004,1.0,1,16000,0,16000,Other - Recreation,Other - Recreation,16000.0,NaN,NaN,NaN,NaN,NaN,1.053706e+06
3373,Nonresidential COS,Other,MAGNOLIA / QUEEN ANNE,47.63644,-122.35784,1974,1.0,1,13157,0,13157,"Fitness Center/Health Club/Gym, Other - Recrea...",Other - Recreation,7583.0,Fitness Center/Health Club/Gym,5574.0,Swimming Pool,0.0,NaN,6.053764e+06
3374,Nonresidential COS,Mixed Use Property,GREATER DUWAMISH,47.52832,-122.32431,1989,1.0,1,14101,0,14101,"Fitness Center/Health Club/Gym, Food Service, ...",Other - Recreation,6601.0,Fitness Center/Health Club/Gym,6501.0,Pre-school/Daycare,484.0,NaN,7.828413e+05


In [801]:
building_consumption.info()

<class 'pandas.DataFrame'>
Index: 1565 entries, 0 to 3375
Data columns (total 20 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   BuildingType                     1565 non-null   str    
 1   PrimaryPropertyType              1565 non-null   str    
 2   Neighborhood                     1565 non-null   str    
 3   Latitude                         1565 non-null   float64
 4   Longitude                        1565 non-null   float64
 5   YearBuilt                        1565 non-null   int64  
 6   NumberofBuildings                1563 non-null   float64
 7   NumberofFloors                   1565 non-null   int64  
 8   PropertyGFATotal                 1565 non-null   int64  
 9   PropertyGFAParking               1565 non-null   int64  
 10  PropertyGFABuilding(s)           1565 non-null   int64  
 11  ListOfAllPropertyUseTypes        1563 non-null   str    
 12  LargestPropertyUseType           155

A réaliser : 
- Une analyse descriptive des données, y compris une explication du sens des colonnes gardées, des arguments derrière la suppression de lignes ou de colonnes, des statistiques descriptives et des visualisations pertinentes.

Qelques pistes d'analyse : 

* Identifier les colonnes avec une majorité de valeurs manquantes ou constantes en utilisant la méthode value_counts() de Pandas
* Mettre en evidence les différences entre les immeubles mono et multi-usages
* Utiliser des pairplots et des boxplots pour faire ressortir les outliers ou des batiments avec des valeurs peu cohérentes d'un point de vue métier 

Pour vous inspirer, ou comprendre l'esprit recherché dans une analyse exploratoire, vous pouvez consulter ce notebook en ligne : https://www.kaggle.com/code/pmarcelino/comprehensive-data-exploration-with-python. Il ne s'agit pas d'un modèle à suivre à la lettre ni d'un template d'analyses attendues pour ce projet. 

# Modélisation 

### Import des modules 

In [802]:
#Selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV, 
    cross_validate,
)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error 
from sklearn.inspection import permutation_importance

#Preprocess
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

#Modèles
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor


### Feature Engineering

A réaliser : Enrichir le jeu de données actuel avec de nouvelles features issues de celles existantes. 

En règle générale : On utilise la méthode .apply() de Pandas pour créer une nouvelle colonne à partir d'une colonne existante. N'hésitez pas à regarder les exemples dans les chapitres de cours donnés en ressource

In [803]:
# CODE FEATURE ENGINEERING

### Préparation des features pour la modélisation

A réaliser :
* Si ce n'est pas déjà fait, supprimer toutes les colonnes peu pertinentes pour la modélisation.
* Tracer la distribution de la cible pour vous familiariser avec l'ordre de grandeur. En cas d'outliers, mettez en place une démarche pour les supprimer.
* Débarrassez-vous des features redondantes en utilisant une matrice de corrélation de Pearson. Pour cela, utiisez la méthode corr() de Pandas, couplé d'un graphique Heatmap de la librairie Seaborn 
* Réalisez différents graphiques pour comprendre le lien entre vos features et la target (boxplots, scatterplots, pairplot si votre nombre de features numériques n'est pas très élevé).
*  Séparez votre jeu de données en un Pandas DataFrame X (ensemble de feautures) et Pandas Series y (votre target).
* Si vous avez des features catégorielles, il faut les encoder pour que votre modèle fonctionne. Les deux méthodes d'encodage à connaitre sont le OneHotEncoder et le LabelEncoder

In [804]:
# CODE PREPARATION DES FEATURES

### Comparaison de différents modèles supervisés

A réaliser :
* Pour chaque algorithme que vous allez tester, vous devez :
    * Réaliser au préalable une séparation en jeu d'apprentissage et jeu de test via une validation croisée.
    * Si les features quantitatives que vous souhaitez utiliser ont des ordres de grandeur très différents les uns des autres, et que vous utilisez un algorithme de regression qui est sensible à cette différence, alors il faut réaliser un scaling (normalisation) de la donnée au préalable.
    * Entrainer le modèle sur le jeu de Train
    * Prédire la cible sur la donnée de test (nous appelons cette étape, l'inférence).
    * Calculer les métriques de performance R2, MAE et RMSE sur le jeu de train et de test.
    * Interpréter les résultats pour juger de la fiabilité de l'algorithme.
* Vous pouvez choisir par exemple de tester un modèle linéaire, un modèle à base d'arbres et un modèle de type SVM
* Déterminer le modèle le plus performant parmi ceux testés.

In [805]:
# CODE COMPARAISON DES MODELES

### Optimisation et interprétation du modèle

A réaliser :
* Reprennez le meilleur algorithme que vous avez sécurisé via l'étape précédente, et réalisez une GridSearch de petite taille sur au moins 3 hyperparamètres.
* Si le meilleur modèle fait partie de la famille des modèles à arbres (RandomForest, GradientBoosting) alors utilisez la fonctionnalité feature importance pour identifier les features les plus impactantes sur la performance du modèle. Sinon, utilisez la méthode Permutation Importance de sklearn.

In [806]:
# CODE OPTIMISATION ET INTERPRETATION DU MODELE